In [1]:
# Phase 0 – Data Alignment | Cell 1: Setup + Inventory

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")

OUT_DIR = PROJECT / "outputs" / "phase0_alignment"
OUT_DATA = OUT_DIR / "data"
OUT_TAB  = OUT_DIR / "tables"
OUT_LOG  = OUT_DIR / "logs"

for d in [OUT_DATA, OUT_TAB, OUT_LOG]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 0 – Data Alignment | Cell 1")
print("=" * 60)
print(f"Timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root : {PROJECT}")
print(f"Outputs      : {OUT_DIR}")

# ---------- Expected paths ----------
paths = {
    "jst_xlsx": PROJECT / "data" / "raw" / "Macrohistory" / "JSTdatasetR6.xlsx",
    "jst_pkl":  PROJECT / "outputs" / "phase2" / "data" / "jst_final_clean.pkl",
    "imf_pkl":  PROJECT / "outputs" / "phase1" / "data" / "imf_weo_raw.pkl",
    "baci_edges": PROJECT / "data" / "raw" / "baci" / "trade_edges_total_2000_2020.parquet",
    "baci_nodes": PROJECT / "data" / "raw" / "baci" / "node_attributes_trade_2000_2020.parquet",
    "bis_long": PROJECT / "data" / "raw" / "bis_credit" / "bis_credit_long.parquet",
    "icio_dir": PROJECT / "data" / "raw" / "oecd_icio",
}

print("\n--- File inventory ---")
inventory = []
for name, p in paths.items():
    exists = p.exists()
    size_mb = (p.stat().st_size / 1e6) if exists and p.is_file() else (
        sum(f.stat().st_size for f in p.glob("*.csv")) / 1e6 if exists and p.is_dir() else 0
    )
    n_files = len(list(p.glob("*_SML.csv"))) if name == "icio_dir" and exists else (1 if exists and p.is_file() else 0)
    status = "FOUND" if exists else "MISSING"
    print(f"  {name:12s} | {status:7s} | size≈{size_mb:8.2f} MB | files={n_files}")
    inventory.append({"source": name, "path": str(p), "exists": exists, "size_mb": round(size_mb, 2), "n_files": n_files})

inv_df = pd.DataFrame(inventory)
inv_df.to_csv(OUT_TAB / "data_inventory.csv", index=False)
print(f"\nSaved → {OUT_TAB / 'data_inventory.csv'}")
print("\nCell 1 completed successfully.")

Phase 0 – Data Alignment | Cell 1
Timestamp    : 2026-09-24 20:52:57
Project Root : D:\univercity\omid_project\economics_project
Outputs      : D:\univercity\omid_project\economics_project\outputs\phase0_alignment

--- File inventory ---
  jst_xlsx     | FOUND   | size≈    1.41 MB | files=1
  jst_pkl      | MISSING | size≈    0.00 MB | files=0
  imf_pkl      | MISSING | size≈    0.00 MB | files=0
  baci_edges   | FOUND   | size≈    9.44 MB | files=1
  baci_nodes   | FOUND   | size≈    0.18 MB | files=1
  bis_long     | FOUND   | size≈    0.04 MB | files=1
  icio_dir     | FOUND   | size≈ 2356.28 MB | files=28

Saved → D:\univercity\omid_project\economics_project\outputs\phase0_alignment\tables\data_inventory.csv

Cell 1 completed successfully.


In [2]:
# Phase 0 – Data Alignment | Cell 2: Country & Year Coverage

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_TAB = PROJECT / "outputs" / "phase0_alignment" / "tables"
OUT_DATA = PROJECT / "outputs" / "phase0_alignment" / "data"
OUT_DATA.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 0 – Data Alignment | Cell 2: Coverage")
print("=" * 60)

# ========== 1) JST ==========
jst = pd.read_excel(PROJECT / "data" / "raw" / "Macrohistory" / "JSTdatasetR6.xlsx")
jst_countries = sorted(jst["country"].dropna().unique().tolist())
jst_iso = sorted(jst["iso"].dropna().unique().tolist()) if "iso" in jst.columns else []
jst_years = (int(jst["year"].min()), int(jst["year"].max()))
print(f"JST     : {len(jst_countries)} countries | years {jst_years[0]}-{jst_years[1]}")
print(f"  countries: {jst_countries}")
print(f"  iso      : {jst_iso}")

# ========== 2) BACI ==========
baci_nodes = pd.read_parquet(PROJECT / "data" / "raw" / "baci" / "node_attributes_trade_2000_2020.parquet")
baci_countries = sorted(baci_nodes["country"].dropna().unique().tolist())
baci_years = (int(baci_nodes["year"].min()), int(baci_nodes["year"].max()))
print(f"\nBACI    : {len(baci_countries)} countries | years {baci_years[0]}-{baci_years[1]}")

# ========== 3) BIS ==========
bis = pd.read_parquet(PROJECT / "data" / "raw" / "bis_credit" / "bis_credit_long.parquet")
bis_countries = sorted(bis["country"].dropna().unique().tolist())
bis_years = (int(bis["year"].min()), int(bis["year"].max()))
print(f"BIS     : {len(bis_countries)} countries | years {bis_years[0]}-{bis_years[1]}")
print(f"  iso2   : {bis_countries}")

# ========== 4) ICIO ==========
icio_dir = PROJECT / "data" / "raw" / "oecd_icio"
icio_files = sorted(icio_dir.glob("*_SML.csv"))
icio_years = sorted(int(f.stem.split("_")[0]) for f in icio_files)
print(f"\nICIO    : {len(icio_files)} files | years {icio_years[0]}-{icio_years[-1]}")

# Sample one ICIO file for country codes
sample = pd.read_csv(icio_dir / f"{icio_years[-1]}_SML.csv", usecols=[0], low_memory=False)
# row labels like AUS_A01, USA_C19, ...
labels = sample.iloc[:, 0].astype(str)
icio_cty = sorted({x.split("_")[0] for x in labels if "_" in x and x.split("_")[0] not in ["TLS", "VA", "OUT"]})
print(f"  countries/regions in ICIO: {len(icio_cty)}")
print(f"  sample: {icio_cty[:15]} ...")

# ========== Mapping helper: JST name/iso → ISO2/ISO3 ==========
# JST iso is usually 3-letter; BIS uses 2-letter; ICIO uses 3-letter
jst_map = jst[["country", "iso"]].drop_duplicates()
print("\n--- JST country ↔ iso ---")
print(jst_map.to_string(index=False))

# Save coverage summary
coverage = pd.DataFrame([
    {"source": "JST",  "n_countries": len(jst_countries), "year_min": jst_years[0], "year_max": jst_years[1], "id_type": "name+iso3"},
    {"source": "BACI", "n_countries": len(baci_countries), "year_min": baci_years[0], "year_max": baci_years[1], "id_type": "iso2"},
    {"source": "BIS",  "n_countries": len(bis_countries), "year_min": bis_years[0], "year_max": bis_years[1], "id_type": "iso2"},
    {"source": "ICIO", "n_countries": len(icio_cty), "year_min": icio_years[0], "year_max": icio_years[-1], "id_type": "iso3"},
])
coverage.to_csv(OUT_TAB / "coverage_by_source.csv", index=False)

# Save lists
pd.DataFrame({"country": jst_countries}).to_csv(OUT_TAB / "jst_countries.csv", index=False)
pd.DataFrame({"iso2": baci_countries}).to_csv(OUT_TAB / "baci_countries_iso2.csv", index=False)
pd.DataFrame({"iso2": bis_countries}).to_csv(OUT_TAB / "bis_countries_iso2.csv", index=False)
pd.DataFrame({"iso3": icio_cty}).to_csv(OUT_TAB / "icio_countries_iso3.csv", index=False)
jst_map.to_csv(OUT_TAB / "jst_country_iso_map.csv", index=False)

print(f"\nSaved coverage tables → {OUT_TAB}")
print("\nCell 2 completed successfully.")

Phase 0 – Data Alignment | Cell 2: Coverage
JST     : 18 countries | years 1870-2020
  countries: ['Australia', 'Belgium', 'Canada', 'Denmark', 'Finland', 'France', 'Germany', 'Ireland', 'Italy', 'Japan', 'Netherlands', 'Norway', 'Portugal', 'Spain', 'Sweden', 'Switzerland', 'UK', 'USA']
  iso      : ['AUS', 'BEL', 'CAN', 'CHE', 'DEU', 'DNK', 'ESP', 'FIN', 'FRA', 'GBR', 'IRL', 'ITA', 'JPN', 'NLD', 'NOR', 'PRT', 'SWE', 'USA']

BACI    : 228 countries | years 2000-2020
BIS     : 18 countries | years 1980-2026
  iso2   : ['AU', 'BE', 'CA', 'CH', 'DE', 'DK', 'ES', 'FI', 'FR', 'GB', 'IE', 'IT', 'JP', 'NL', 'NO', 'PT', 'SE', 'US']

ICIO    : 28 files | years 1995-2022
  countries/regions in ICIO: 81
  sample: ['AGO', 'ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGD', 'BGR', 'BLR', 'BRA', 'BRN', 'CAN', 'CHE', 'CHL', 'CHN'] ...

--- JST country ↔ iso ---
    country iso
  Australia AUS
    Belgium BEL
     Canada CAN
Switzerland CHE
    Germany DEU
    Denmark DNK
      Spain ESP
    Finland FIN
     F

In [3]:
# Phase 0 – Data Alignment | Cell 3: Country Crosswalk + Common Years

from pathlib import Path
import pandas as pd

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_TAB = PROJECT / "outputs" / "phase0_alignment" / "tables"
OUT_DATA = PROJECT / "outputs" / "phase0_alignment" / "data"

print("=" * 60)
print("Phase 0 – Data Alignment | Cell 3: Crosswalk")
print("=" * 60)

# Manual ISO2 ↔ ISO3 for the 18 JST countries
crosswalk = pd.DataFrame([
    {"country_jst": "Australia",    "iso3": "AUS", "iso2": "AU"},
    {"country_jst": "Belgium",      "iso3": "BEL", "iso2": "BE"},
    {"country_jst": "Canada",       "iso3": "CAN", "iso2": "CA"},
    {"country_jst": "Switzerland",  "iso3": "CHE", "iso2": "CH"},
    {"country_jst": "Germany",      "iso3": "DEU", "iso2": "DE"},
    {"country_jst": "Denmark",      "iso3": "DNK", "iso2": "DK"},
    {"country_jst": "Spain",        "iso3": "ESP", "iso2": "ES"},
    {"country_jst": "Finland",      "iso3": "FIN", "iso2": "FI"},
    {"country_jst": "France",       "iso3": "FRA", "iso2": "FR"},
    {"country_jst": "UK",           "iso3": "GBR", "iso2": "GB"},
    {"country_jst": "Ireland",      "iso3": "IRL", "iso2": "IE"},
    {"country_jst": "Italy",        "iso3": "ITA", "iso2": "IT"},
    {"country_jst": "Japan",        "iso3": "JPN", "iso2": "JP"},
    {"country_jst": "Netherlands",  "iso3": "NLD", "iso2": "NL"},
    {"country_jst": "Norway",       "iso3": "NOR", "iso2": "NO"},
    {"country_jst": "Portugal",     "iso3": "PRT", "iso2": "PT"},
    {"country_jst": "Sweden",       "iso3": "SWE", "iso2": "SE"},
    {"country_jst": "USA",          "iso3": "USA", "iso2": "US"},
])

# Load coverage lists
icio_cty = set(pd.read_csv(OUT_TAB / "icio_countries_iso3.csv")["iso3"])
baci_cty = set(pd.read_csv(OUT_TAB / "baci_countries_iso2.csv")["iso2"])
bis_cty  = set(pd.read_csv(OUT_TAB / "bis_countries_iso2.csv")["iso2"])

crosswalk["in_ICIO"] = crosswalk["iso3"].isin(icio_cty)
crosswalk["in_BACI"] = crosswalk["iso2"].isin(baci_cty)
crosswalk["in_BIS"]  = crosswalk["iso2"].isin(bis_cty)
crosswalk["in_all"]  = crosswalk["in_ICIO"] & crosswalk["in_BACI"] & crosswalk["in_BIS"]

print("--- Country Crosswalk (18 JST) ---")
print(crosswalk.to_string(index=False))
print(f"\nIn all sources: {crosswalk['in_all'].sum()} / 18")

crosswalk.to_csv(OUT_TAB / "country_crosswalk_jst18.csv", index=False)
crosswalk.to_pickle(OUT_DATA / "country_crosswalk_jst18.pkl")

# Common year window
# JST: 1870-2020 | BACI: 2000-2020 | BIS: 1980-2026 | ICIO: 1995-2022
common_start = max(2000, 1995, 1980)  # practical overlap start
common_end   = min(2020, 2020, 2022)  # limited by JST/BACI
print(f"\nRecommended common year window: {common_start} – {common_end}")

year_plan = pd.DataFrame({
    "source": ["JST", "BACI", "BIS", "ICIO", "COMMON"],
    "year_min": [1870, 2000, 1980, 1995, common_start],
    "year_max": [2020, 2020, 2026, 2022, common_end],
})
print(year_plan.to_string(index=False))
year_plan.to_csv(OUT_TAB / "year_coverage_plan.csv", index=False)

print("\nCell 3 completed successfully.")

Phase 0 – Data Alignment | Cell 3: Crosswalk
--- Country Crosswalk (18 JST) ---
country_jst iso3 iso2  in_ICIO  in_BACI  in_BIS  in_all
  Australia  AUS   AU     True     True    True    True
    Belgium  BEL   BE     True     True    True    True
     Canada  CAN   CA     True     True    True    True
Switzerland  CHE   CH     True     True    True    True
    Germany  DEU   DE     True     True    True    True
    Denmark  DNK   DK     True     True    True    True
      Spain  ESP   ES     True     True    True    True
    Finland  FIN   FI     True     True    True    True
     France  FRA   FR     True     True    True    True
         UK  GBR   GB     True     True    True    True
    Ireland  IRL   IE     True     True    True    True
      Italy  ITA   IT     True     True    True    True
      Japan  JPN   JP     True     True    True    True
Netherlands  NLD   NL     True     True    True    True
     Norway  NOR   NO     True     True    True    True
   Portugal  PRT   PT   

In [4]:
# Phase 0 – Data Alignment | Cell 4: Finalize aligned subsets

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_TAB  = PROJECT / "outputs" / "phase0_alignment" / "tables"
OUT_DATA = PROJECT / "outputs" / "phase0_alignment" / "data"

print("=" * 60)
print("Phase 0 – Data Alignment | Cell 4: Finalize")
print("=" * 60)

crosswalk = pd.read_pickle(OUT_DATA / "country_crosswalk_jst18.pkl")
iso2_set = set(crosswalk["iso2"])
iso3_set = set(crosswalk["iso3"])
YEAR_MIN, YEAR_MAX = 2000, 2020

# ----- JST filtered -----
jst = pd.read_excel(PROJECT / "data" / "raw" / "Macrohistory" / "JSTdatasetR6.xlsx")
jst_f = jst[(jst["year"] >= YEAR_MIN) & (jst["year"] <= YEAR_MAX)].copy()
jst_f = jst_f.merge(crosswalk[["country_jst", "iso2", "iso3"]],
                    left_on="country", right_on="country_jst", how="left")
print(f"JST filtered : {jst_f.shape} | countries={jst_f['iso3'].nunique()} | years={jst_f['year'].min()}-{jst_f['year'].max()}")

# ----- BACI edges filtered -----
baci = pd.read_parquet(PROJECT / "data" / "raw" / "baci" / "trade_edges_total_2000_2020.parquet")
baci_f = baci[
    (baci["year"] >= YEAR_MIN) & (baci["year"] <= YEAR_MAX) &
    (baci["source_iso2"].isin(iso2_set)) & (baci["target_iso2"].isin(iso2_set))
].copy()
print(f"BACI edges   : {baci_f.shape} | src={baci_f['source_iso2'].nunique()} | tgt={baci_f['target_iso2'].nunique()}")

# ----- BIS filtered (annual mean from quarterly) -----
bis = pd.read_parquet(PROJECT / "data" / "raw" / "bis_credit" / "bis_credit_long.parquet")
bis_f = bis[
    (bis["year"] >= YEAR_MIN) & (bis["year"] <= YEAR_MAX) &
    (bis["country"].isin(iso2_set))
].copy()
# Prefer: Private non-financial (P), All lenders (A), Market value (M), unit 770
bis_core = bis_f[
    (bis_f["borrowers"] == "P") &
    (bis_f["lenders"].astype(str).str.startswith("A")) &
    (bis_f["valuation"] == "M")
].copy()
if bis_core.empty:
    bis_core = bis_f.copy()
bis_annual = bis_core.groupby(["year", "country"], as_index=False)["value"].mean()
bis_annual = bis_annual.rename(columns={"country": "iso2", "value": "credit_private_pct_gdp"})
print(f"BIS annual   : {bis_annual.shape} | countries={bis_annual['iso2'].nunique()}")

# ----- Save aligned panels -----
jst_f.to_pickle(OUT_DATA / "jst_aligned_2000_2020.pkl")
jst_f.to_csv(OUT_TAB / "jst_aligned_2000_2020.csv", index=False)

baci_f.to_pickle(OUT_DATA / "baci_edges_jst18_2000_2020.pkl")
baci_f.to_csv(OUT_TAB / "baci_edges_jst18_2000_2020.csv", index=False)

bis_annual.to_pickle(OUT_DATA / "bis_credit_jst18_annual.pkl")
bis_annual.to_csv(OUT_TAB / "bis_credit_jst18_annual.csv", index=False)

crosswalk.to_csv(OUT_TAB / "country_crosswalk_jst18.csv", index=False)

print("\nSaved aligned datasets:")
print(f"  {OUT_DATA / 'jst_aligned_2000_2020.pkl'}")
print(f"  {OUT_DATA / 'baci_edges_jst18_2000_2020.pkl'}")
print(f"  {OUT_DATA / 'bis_credit_jst18_annual.pkl'}")

print("\n--- Alignment Summary ---")
print(f"Countries (JST18): 18/18 in all sources")
print(f"Common years     : {YEAR_MIN}-{YEAR_MAX}")
print(f"BACI internal edges among JST18: {len(baci_f):,}")
print(f"BIS annual rows: {len(bis_annual):,}")

print("\nCell 4 completed successfully.")

Phase 0 – Data Alignment | Cell 4: Finalize
JST filtered : (378, 62) | countries=18 | years=2000-2020
BACI edges   : (6426, 14) | src=18 | tgt=18
BIS annual   : (315, 3) | countries=15

Saved aligned datasets:
  D:\univercity\omid_project\economics_project\outputs\phase0_alignment\data\jst_aligned_2000_2020.pkl
  D:\univercity\omid_project\economics_project\outputs\phase0_alignment\data\baci_edges_jst18_2000_2020.pkl
  D:\univercity\omid_project\economics_project\outputs\phase0_alignment\data\bis_credit_jst18_annual.pkl

--- Alignment Summary ---
Countries (JST18): 18/18 in all sources
Common years     : 2000-2020
BACI internal edges among JST18: 6,426
BIS annual rows: 315

Cell 4 completed successfully.
